<a href="https://colab.research.google.com/github/Bix4UMD/BIOI611_lab/blob/main/docs/BIOI611_DAVID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Functional enrichment with DAVID

[DAVID](https://davidbioinformatics.nih.gov/) tests whether functional annotations occur more often in a submitted gene list than expected from a suitable background. It combines Gene Ontology, pathways, protein domains, sequence features, and other annotation sources in one interface.

> **Version used in this lab:** DAVID 2025 system release (March 2026), Knowledgebase v2026_2 (released October 6, 2026), accessed November 3, 2026. DAVID's interface and knowledgebase change over time, so record both versions in your report.

## Learning objectives

By the end of this lab, you should be able to:

1. create a foreground gene list and a matching tested-gene background;
2. upload WormBase identifiers using the current taxonomy-based DAVID interface;
3. check identifier mapping before enrichment;
4. distinguish the Summary, Chart, Table, and Clustering reports; and
5. interpret EASE and Benjamini values without treating enrichment as proof of mechanism.

## 1. Prepare the *C. elegans* input lists

The worked example uses `BIOI_bulkRNAseq_SE_DESeq2_res_filter.xlsx`, which contains one DESeq2-results row per WormBase gene identifier. The spreadsheet filter is reproduced exactly:

- **Foreground:** `log2FoldChange > 2` and `padj < 0.05` (strictly greater than 2), giving **1,039 unique genes**.
- **Background upload:** every gene with a nonmissing `padj`, giving **18,709 unique WormBase IDs**. These are the genes eligible to enter a foreground defined by an adjusted-P-value cutoff.

The 1,039-gene foreground is below DAVID's current 3,000-gene limit for Functional Annotation Clustering.

For this example, positive log2 fold change is labeled “Day 7 upregulated.” Before reusing the workflow for another contrast, confirm which condition is the numerator in your DESeq2 results. The workbook alone does not encode that direction.

Ready-to-upload files:

- [Foreground: 1,039 upregulated WormBase IDs](https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/BIOI_bulkRNAseq_SE_DAVID_upregulated_WormBase_IDs.txt)
- [Background: 18,709 tested WormBase IDs](https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/BIOI_bulkRNAseq_SE_DAVID_background_nonmissing_padj_WormBase_IDs.txt)

> A whole-genome background is not automatically appropriate. The background should represent the genes that could have been selected by the experiment and upstream analysis. Changing the background can change both the enrichment estimates and the conclusions.

In [ ]:
# Optional: reproduce the two text files from the supplied workbook.
# In Colab, upload BIOI_bulkRNAseq_SE_DESeq2_res_filter.xlsx first.
if (!requireNamespace("readxl", quietly = TRUE)) {
  install.packages("readxl", repos = "https://cloud.r-project.org")
}

res <- readxl::read_excel(
  "BIOI_bulkRNAseq_SE_DESeq2_res_filter.xlsx",
  na = c("", "NA")
)
names(res)[1] <- "WormBase_gene_id"
res$WormBase_gene_id <- trimws(as.character(res$WormBase_gene_id))

foreground_ids <- unique(res$WormBase_gene_id[
  !is.na(res$log2FoldChange) & !is.na(res$padj) &
    res$log2FoldChange > 2 & res$padj < 0.05
])
background_ids <- unique(res$WormBase_gene_id[!is.na(res$padj)])

# These expected counts apply to the supplied workbook.
stopifnot(
  length(foreground_ids) == 1039,
  length(background_ids) == 18709,
  all(grepl("^WBGene[0-9]{8}$", foreground_ids)),
  all(grepl("^WBGene[0-9]{8}$", background_ids)),
  all(foreground_ids %in% background_ids)
)

writeLines(foreground_ids, "BIOI_bulkRNAseq_SE_DAVID_upregulated_WormBase_IDs.txt")
writeLines(background_ids, "BIOI_bulkRNAseq_SE_DAVID_background_nonmissing_padj_WormBase_IDs.txt")

c(foreground = length(foreground_ids), background = length(background_ids))

## 2. Upload the foreground in the current DAVID interface

1. Open [DAVID Start Analysis](https://davidbioinformatics.nih.gov/workspace.html).
2. In **List Manager**, choose **Upload User List**.
3. Paste the one-ID-per-line contents (the illustrated workflow), with no header. If you instead choose **Upload file**, select the first column as **Identifier** and uncheck **File has header row?** because the prepared text file is headerless.
4. Choose **Caenorhabditis elegans (6239)** as the taxonomy.
5. Use a descriptive name such as `Day 7 upregulated (LFC > 2; DESeq2 padj < 0.05)`, then select **Upload List**.

The current interface resolves `WBGene…` identifiers automatically within the selected taxonomy; it does not normally ask for the old `WORMBASE_GENE_ID` choice. The IDs in this exercise are stable WormBase identifiers such as `WBGene00000001`, not gene symbols.

> **Terminology note:** The captured screenshots use `FDR < 0.05` in the list name as shorthand for the upstream DESeq2 `padj < 0.05` filter. That label is not DAVID's separate FDR output column. Use `DESeq2 padj` in future list names.

<figure>
  <img src="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_upload_worm_list_annotated.png" alt="Annotated current DAVID Upload User List dialog showing the WormBase foreground, list name, and Caenorhabditis elegans taxonomy" style="width:100%; max-width:1100px;">
  <figcaption><strong>Figure 1.</strong> Current Upload User List dialog for the 1,039-gene foreground. Orange labels identify the decisions that must be checked before upload. <a href="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_upload_worm_list_annotated.png" target="_blank">Open full-size image</a>.</figcaption>
</figure>

## 3. Check identifier mapping before enrichment

Open **Gene List Report** and compare the mapped count with the number of submitted IDs. A low mapping rate can produce a misleading enrichment result even when DAVID completes the analysis.

| List | Submitted unique IDs | Mapped submitted IDs | Effective DAVID genes | Mapping rate |
|---|---:|---:|---:|---:|
| Foreground | 1,039 | 1,039 | 1,039 | 100.00% |
| Background | 18,709 | 18,702 | 18,702 | 99.96% |

DAVID returned 18,703 background mapping rows because `WBGene00000464` mapped to two DAVID genes. In the other direction, `WBGene00022661` and `WBGene00022662` collapsed to the same DAVID gene (`191228`). After DAVID-gene de-duplication, the effective background population is therefore **18,702 DAVID genes**, not 18,709. Report both the submitted-ID and effective-population counts, and keep the seven unmapped IDs in the audit trail.

<details><summary>Seven background IDs not mapped in Knowledgebase v2026_2</summary>

`WBGene00018587`, `WBGene00014472`, `WBGene00010958`, `WBGene00014454`, `WBGene00271812`, `WBGene00010959`, and `WBGene00007501`.

</details>

<figure>
  <img src="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_gene_list_report_annotated.png" alt="Annotated current DAVID Gene List Report showing 1,039 mapped WormBase identifiers and the main mapping quality-control columns" style="width:100%; max-width:1100px;">
  <figcaption><strong>Figure 2.</strong> Gene List Report for the foreground. Check the list name, record count, submitted WormBase ID, mapped gene, and species. <a href="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_gene_list_report_annotated.png" target="_blank">Open full-size image</a>.</figcaption>
</figure>

## 4. Upload and confirm the custom background

Use **Background Manager → Upload User Background** to upload the 18,709-ID background with the same *C. elegans* taxonomy. Before interpreting a report, confirm that DAVID shows both:

- **Current Gene List:** the 1,039-gene foreground; and
- **Current Population:** the tested background named for its 18,709 submitted IDs; mapping and de-duplication give an effective population of 18,702 DAVID genes.

If the Current Population instead shows the default genome, stop and select the custom background. DAVID explicitly recommends a population that represents the genes from which the submitted list was derived; an inappropriate genome background can inflate apparent enrichment.

## 5. Read the DAVID reports

### Functional Annotation Summary

The Summary organizes annotation sources and reports annotation **coverage**. A long bar or high percentage means that many submitted genes have that type of annotation; it is not evidence of enrichment. Choose focused sources appropriate to the question—for example GO BP/CC/MF Direct and selected pathway databases—rather than treating every source as independent evidence.

<figure>
  <img src="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_summary_annotated.png" alt="Annotated current DAVID Functional Annotation Summary showing the foreground, custom background, category tabs, annotation coverage, and report buttons" style="width:100%; max-width:1100px;">
  <figcaption><strong>Figure 3.</strong> Functional Annotation Summary using the custom background: 18,709 submitted IDs and an effective population of 18,702 mapped, de-duplicated DAVID genes. <a href="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_summary_annotated.png" target="_blank">Open full-size image</a>.</figcaption>
</figure>

### Functional Annotation Chart

The Chart is the term-centric enrichment report. In the current interface, the default display filters are **EASE ≤ 0.1** and **Count ≥ 2**. For this saved session, those settings produced 224 chart records; that number can change when categories, filters, the background, or DAVID's knowledgebase change.

- **P-Value** is DAVID's EASE score (modified one-sided Fisher test).
- **Benjamini** is the Benjamini–Hochberg adjusted value within the relevant annotation-source family and should be the primary multiplicity-aware evidence column.
- **Genes**, **Count**, and the gene links show which foreground genes support a term.
- **RT** opens related terms and helps reveal redundant annotations.

<figure>
  <img src="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_chart_annotated.png" alt="Annotated current DAVID Functional Annotation Chart showing custom background, filters, term columns, gene coverage, EASE P-Value, and Benjamini value" style="width:100%; max-width:1100px;">
  <figcaption><strong>Figure 4.</strong> Chart report for the 1,039-gene foreground. The visible leading terms include overlapping F-box, cuticle/collagen, immune-response, and cytochrome-P450 annotations; treat these as version- and background-dependent hypotheses, not independent confirmations or causal conclusions. <a href="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_chart_annotated.png" target="_blank">Open full-size image</a>.</figcaption>
</figure>

### Functional Annotation Table

The Table is gene-centric: it is useful for looking up annotations assigned to individual genes, but it is not an enrichment-statistics report.

### Functional Annotation Clustering

Clustering groups annotations that share genes, reducing some of the redundancy in the Chart. The current default shown here uses **Medium** classification stringency and an EASE threshold of 1.

A cluster's **Enrichment Score** summarizes its member EASE values on a negative-log scale; it is **not** an adjusted P value. Judge statistical evidence using the member terms' Benjamini values, then inspect the shared genes and biological coherence. In this session, the leading groups summarize F-box, cuticle/collagen, and cytochrome-P450-related annotations.

<figure>
  <img src="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_clustering_annotated.png" alt="Annotated current DAVID Functional Annotation Clustering report showing custom background, clustering settings, cluster enrichment score, member terms, and term-level Benjamini values" style="width:100%; max-width:1100px;">
  <figcaption><strong>Figure 5.</strong> Functional Annotation Clustering report. The Group Enrichment Score ranks related-term groups but does not replace term-level multiple-testing correction. <a href="https://raw.githubusercontent.com/Bix4UMD/BIOI611_lab/main/docs/assets/david/david_annotation_clustering_annotated.png" target="_blank">Open full-size image</a>.</figcaption>
</figure>

## 6. Understand Fisher's exact test and the EASE score

For one annotation term, DAVID compares the input list with the selected population using a 2 × 2 table:

| | In the term | Not in the term |
|---|---:|---:|
| Input list | List Hits (LH) | List Total − LH |
| Other genes in the population | Population Hits − LH | Remaining population genes |

A one-sided Fisher test asks whether the observed overlap is at least as large as expected under independence. DAVID's **EASE score** removes one term hit from the modified input list: List Hits becomes `LH − 1`, list non-hits stay unchanged, and the modified list total becomes `LT − 1`. This deliberately penalizes terms supported by few genes. If a term has only one List Hit, its EASE score provides no enrichment evidence.

DAVID calculates each Chart row from the genes annotated in that row's annotation source. Its displayed List Total (LT) and Population Total (PT) can therefore be smaller than the overall 1,039-gene foreground and 18,702-gene effective population.

A small raw value supports over-representation under the chosen list, population, and annotation definitions. It does not show that a pathway is activated, causal, or biologically important by itself. Because DAVID tests many correlated terms, interpret **Benjamini** values alongside gene counts, effect sizes, and the underlying genes. A **Benjamini value** above 0.05 means the analysis did not provide strong multiplicity-adjusted evidence at that threshold; it is not proof of no biological effect.

In [ ]:
# Conceptual example: a 300-gene list, a 30,000-gene population,
# 40 population genes in a term, and 3 list genes in that term.
LH <- 3
LT <- 300
PH <- 40
PT <- 30000

table_fisher <- matrix(
  c(LH, LT - LH, PH - LH, (PT - PH) - (LT - LH)),
  nrow = 2, byrow = TRUE,
  dimnames = list(
    c("Input list", "Other population genes"),
    c("In term", "Not in term")
  )
)

# EASE removes one list hit; list non-hits stay fixed, so modified LT = LT - 1.
LH_ease <- LH - 1
table_ease <- matrix(
  c(LH_ease, LT - LH, PH - LH_ease, (PT - PH) - (LT - LH)),
  nrow = 2, byrow = TRUE,
  dimnames = dimnames(table_fisher)
)

data.frame(
  test = c("one-sided Fisher", "EASE-style modified Fisher"),
  p_value = c(
    fisher.test(table_fisher, alternative = "greater")$p.value,
    fisher.test(table_ease, alternative = "greater")$p.value
  )
)
# Expected values are approximately 0.00744253 and 0.06027361.
# A Benjamini value cannot be reproduced from one term alone; it depends
# on the other terms in DAVID's relevant annotation-source family.

## 7. Interpretation and reproducibility checklist

Before reporting a result:

- [ ] Record the foreground rule, contrast direction, foreground size, and ID type.
- [ ] Record the background rule, submitted-ID count, and effective mapped population; verify **Current Population** in DAVID.
- [ ] Report foreground and background mapping rates.
- [ ] Record the DAVID system version, knowledgebase version, access date, selected annotation sources, and display thresholds.
- [ ] Use Benjamini values for multiplicity-aware interpretation; do not rank evidence by raw EASE alone.
- [ ] Inspect the genes behind each leading term and note overlapping/redundant terms.
- [ ] Describe enriched annotations as associations or hypotheses, not as proof of pathway activity or causation.
- [ ] Export the Chart and Clustering tables using **CSV** or **Excel** and retain the two uploaded ID files. DAVID exports the current filtered and sorted view, so record those settings.

The exact files, checksums, mapping audit, screenshot provenance, and known limits for this worked example are recorded in the [DAVID asset README](https://github.com/Bix4UMD/BIOI611_lab/blob/main/docs/assets/david/README.md).

## References

- [DAVID current documentation](https://davidbioinformatics.nih.gov/documentation/)
- [List Manager guide](https://davidbioinformatics.nih.gov/documentation/tools/listManager.html)
- [Gene-enrichment principles and background selection](https://davidbioinformatics.nih.gov/documentation/tools/tech-gene-enrichment.html)
- [Functional Annotation Chart documentation](https://davidbioinformatics.nih.gov/documentation/tools/chartReport.html)
- [Functional Annotation Clustering documentation](https://davidbioinformatics.nih.gov/documentation/tools/term2term.html)
- [DAVID system releases](https://davidbioinformatics.nih.gov/documentation/tools/release.html) and [knowledgebase releases](https://davidbioinformatics.nih.gov/documentation/tools/kbUpdate.html)
- Sherman BT, et al. DAVID: a web server for functional annotation and functional enrichment analysis of gene lists (2025 update). *Nucleic Acids Research*. 2026. [doi:10.1093/nar/gkag470](https://doi.org/10.1093/nar/gkag470)